####Ex 1→ Setup + Write your first Delta Table
Delta Lake is the default and only storage format you should use on Databricks. Every production pipeline writes Delta. This exercise creates the base employee table we'll use across all exercises.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

# Employee sales table - used across all exercises
# CREATE DATAFRAME
schema = StructType([
    StructField("emp_id",   IntegerType(), False),
    StructField("name",     StringType(),  False),
    StructField("dept",     StringType(),  True),
    StructField("city",     StringType(),  True),
    StructField("salary",   DoubleType(),  True),
    StructField("status",   StringType(),  True),
])
data = [
    (1,  "Alice",   "Engineering", "Delhi",     95000.0,  "Active"),
    (2,  "Bob",     "Marketing",   "Mumbai",    72000.0,  "Active"),
    (3,  "Charlie", "Engineering", "Bangalore", 88000.0,  "Active"),
    (4,  "Diana",   "HR",          "Delhi",     65000.0,  "Active"),
    (5,  "Eve",     "Marketing",   "Mumbai",    78000.0,  "Active"),
    (6,  "Frank",   "Engineering", "Chennai",  102000.0,  "Active"),
    (7,  "Grace",   "HR",          "Bangalore", 61000.0,  "Active"),
    (8,  "Henry",   "Marketing",   "Delhi",     74000.0,  "Active"),
]

df = spark.createDataFrame(data,schema)
df.display()

# WRITE IT AS DELTA TABLE - the standard on Databricks
df.write.format("delta") \
        .mode("overwrite") \
        .save("/Volumes/pyspark_course/data_tables/delta_volume")

# VERIFY DELTA TABLE
spark.sql("SELECT * FROM employees").display()

# CHECK DELTA META-DATA
spark.sql("DESCRIBE DETAIL employees") \
     .select("format","numFiles","sizeInBytes","location") \
     .display()

# CHECK TABLE PROPERTIES
spark.sql("DESCRIBE EXTENDED employees").display()

emp_id,name,dept,city,salary,status
1,Alice,Engineering,Delhi,95000.0,Active
2,Bob,Marketing,Mumbai,72000.0,Active
3,Charlie,Engineering,Bangalore,88000.0,Active
4,Diana,HR,Delhi,65000.0,Active
5,Eve,Marketing,Mumbai,78000.0,Active
6,Frank,Engineering,Chennai,102000.0,Active
7,Grace,HR,Bangalore,61000.0,Active
8,Henry,Marketing,Delhi,74000.0,Active


emp_id,name,dept,city,salary,status
2,Bob,Marketing,Mumbai,72000.0,Active
4,Diana,HR,Delhi,65000.0,Active
5,Eve,Marketing,Mumbai,78000.0,Active
7,Grace,HR,Bangalore,61000.0,Active
8,Henry,Marketing,Delhi,74000.0,Active
1,Alice,Engineering,Delhi,108000.0,Active
3,Charlie,Engineering,Hyderabad,92000.0,Active
6,Frank,Engineering,Chennai,105000.0,Active
9,Ivy,Engineering,Pune,91000.0,Active
10,Jack,HR,Chennai,58000.0,Active


format,numFiles,sizeInBytes,location
delta,1,2066,


col_name,data_type,comment
emp_id,int,null
name,string,null
dept,string,null
city,string,null
salary,double,null
status,string,null
,,
# Delta Statistics Columns,,
Column Names,"city, name, dept, salary, status, emp_id",
Column Selection Method,first-32,


####Ex 2→ Read Delta Table - 3 ways
Know all three ways to read a Delta table. In production you'll use all three depending on the context — Python pipeline, SQL analyst query, or programmatic Delta API access.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

# WAY 1: spark.table()- cleanest, uses metastore name
df1 = spark.table("employees")
df.display()

# WAY 2: spark.read.format("delta") -when you have a path
df2 = spark.read.format("delta") \
                .load("/Volumes/pyspark_course/data_tables/delta_volume")
df2.display()

# WAY 3: spark.sql() - pure SQL
spark.sql("SELECT * FROM employees WHERE dept = 'Engineering'").display()

# WAY 4: DeltaTable API - for programmatic operations (MERGE,history)
delta_tb1 = DeltaTable.forName(spark, "employees")
delta_tb1.toDF().display()

emp_id,name,dept,city,salary,status
1,Alice,Engineering,Delhi,95000.0,Active
2,Bob,Marketing,Mumbai,72000.0,Active
3,Charlie,Engineering,Bangalore,88000.0,Active
4,Diana,HR,Delhi,65000.0,Active
5,Eve,Marketing,Mumbai,78000.0,Active
6,Frank,Engineering,Chennai,102000.0,Active
7,Grace,HR,Bangalore,61000.0,Active
8,Henry,Marketing,Delhi,74000.0,Active


emp_id,name,dept,city,salary,status
1,Alice,Engineering,Delhi,95000.0,Active
2,Bob,Marketing,Mumbai,72000.0,Active
3,Charlie,Engineering,Bangalore,88000.0,Active
4,Diana,HR,Delhi,65000.0,Active
5,Eve,Marketing,Mumbai,78000.0,Active
6,Frank,Engineering,Chennai,102000.0,Active
7,Grace,HR,Bangalore,61000.0,Active
8,Henry,Marketing,Delhi,74000.0,Active


emp_id,name,dept,city,salary,status
1,Alice,Engineering,Delhi,95000.0,Active
3,Charlie,Engineering,Bangalore,88000.0,Active
6,Frank,Engineering,Chennai,102000.0,Active


emp_id,name,dept,city,salary,status
1,Alice,Engineering,Delhi,95000.0,Active
2,Bob,Marketing,Mumbai,72000.0,Active
3,Charlie,Engineering,Bangalore,88000.0,Active
4,Diana,HR,Delhi,65000.0,Active
5,Eve,Marketing,Mumbai,78000.0,Active
6,Frank,Engineering,Chennai,102000.0,Active
7,Grace,HR,Bangalore,61000.0,Active
8,Henry,Marketing,Delhi,74000.0,Active


####Ex 3→ INSERT,UPDATE,DELETE on Delta Tables
Delta supports full DML — INSERT, UPDATE, DELETE — with ACID guarantees. Unlike raw Parquet, a failed UPDATE doesn't leave your table in a corrupted half-written state. Delta either fully commits or fully rolls back.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

spark.sql("SHOW TABLES").display()
spark.sql("SELECT * FROM employees").display()

# INSERT - add new rows
spark.sql("""
      INSERT INTO employees VALUES 
      (9, 'Ivy', 'Engineering', 'Pune', 91000.0, 'Active'),
      (10, 'Jack', 'HR', 'Chennai', 58000.0, 'Active')    
""")

print("After INSERT: ")
spark.sql("SELECT * FROM employees").display()

# UPDATE - modify existing rows → give Engineering dept a 10% hike
spark.sql("""
      UPDATE employees
      SET salary = ROUND(salary * 1.10, 2)
      WHERE dept = 'Engineering'
""")

print("Engineers salaries after hike: ")
spark.sql("""
      SELECT *
      FROM employees
      WHERE dept = 'Engineering'
      ORDER BY salary DESC    
""").display()

# DELETE - remove rows
spark.sql("""
     DELETE
     FROM employees
     WHERE emp_id IN (9,10,11,12,13,14,15,16,17,18,19,20)       
""")
print("Deleted all the rows we hace made extra for this demo:")
spark.sql("SELECT * FROM employees").display()

emp_id,name,dept,city,salary,status
1,Alice,Engineering,Delhi,126445.0,Active
3,Charlie,Engineering,Bangalore,117128.0,Active
6,Frank,Engineering,Chennai,135762.0,Active
2,Bob,Marketing,Mumbai,72000.0,Active
4,Diana,HR,Delhi,65000.0,Active
5,Eve,Marketing,Mumbai,78000.0,Active
7,Grace,HR,Bangalore,61000.0,Active
8,Henry,Marketing,Delhi,74000.0,Active
9,Ivy,Engineering,Pune,91000.0,Active
10,Jack,HR,Chennai,58000.0,Active


####Ex 4→ Overwrite and append - write modes on Delta Tables
Delta's overwrite modes are smarter than raw Parquet. replaceWhere lets you overwrite just one partition's worth of data without touching the rest — the most important write pattern in incremental pipelines.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

# Regular overwrite — replaces ENTIRE table
new_full_data = spark.createDataFrame([
    (1, "Alice",   "Engineering", "Delhi",    104500.0, "Active"),
    (2, "Bob",     "Marketing",   "Mumbai",    72000.0, "Active"),
    (3, "Charlie", "Engineering", "Bangalore", 96800.0, "Active"),
], schema)

new_full_data.write.format("delta") \
                   .mode("overwrite") \
                   .saveAsTable("employees")

spark.sql("SELECT * FROM employees").display() 

# Restore full dataset first
df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("employees")

spark.sql("SELECT * FROM employees").display()

# APPEND - add rows without touching existing data
new_employees = spark.createDataFrame([
    (9,  "Ivy",  "Engineering", "Pune",   91000.0, "Active"),
    (10, "Jack", "HR",          "Chennai", 58000.0, "Active"),
], schema)

new_employees.write.format("delta") \
                   .mode("append") \
                   .saveAsTable("employees")

spark.sql("SELECT * FROM employees").display()

# replaceWhere — overwrite ONLY matching rows, keep the rest
  ## Use case: re-process just the Engineering department
updated_eng = spark.createDataFrame([
    (1, "Alice",   "Engineering", "Delhi",    110000.0, "Active"),
    (3, "Charlie", "Engineering", "Bangalore", 99000.0, "Active"),
    (6, "Frank",   "Engineering", "Chennai",  115000.0, "Active"),
], schema)

updated_eng.write.format("delta") \
                 .mode("overwrite") \
                 .option("replaceWhere", "dept = 'Engineering'") \
                 .saveAsTable("employees")

spark.sql("SELECT * FROM employees").display()

emp_id,name,dept,city,salary,status
1,Alice,Engineering,Delhi,104500.0,Active
2,Bob,Marketing,Mumbai,72000.0,Active
3,Charlie,Engineering,Bangalore,96800.0,Active


emp_id,name,dept,city,salary,status
1,Alice,Engineering,Delhi,95000.0,Active
2,Bob,Marketing,Mumbai,72000.0,Active
3,Charlie,Engineering,Bangalore,88000.0,Active
4,Diana,HR,Delhi,65000.0,Active
5,Eve,Marketing,Mumbai,78000.0,Active
6,Frank,Engineering,Chennai,102000.0,Active
7,Grace,HR,Bangalore,61000.0,Active
8,Henry,Marketing,Delhi,74000.0,Active


emp_id,name,dept,city,salary,status
1,Alice,Engineering,Delhi,95000.0,Active
2,Bob,Marketing,Mumbai,72000.0,Active
3,Charlie,Engineering,Bangalore,88000.0,Active
4,Diana,HR,Delhi,65000.0,Active
5,Eve,Marketing,Mumbai,78000.0,Active
6,Frank,Engineering,Chennai,102000.0,Active
7,Grace,HR,Bangalore,61000.0,Active
8,Henry,Marketing,Delhi,74000.0,Active
9,Ivy,Engineering,Pune,91000.0,Active
10,Jack,HR,Chennai,58000.0,Active


emp_id,name,dept,city,salary,status
2,Bob,Marketing,Mumbai,72000.0,Active
4,Diana,HR,Delhi,65000.0,Active
5,Eve,Marketing,Mumbai,78000.0,Active
7,Grace,HR,Bangalore,61000.0,Active
8,Henry,Marketing,Delhi,74000.0,Active
1,Alice,Engineering,Delhi,110000.0,Active
3,Charlie,Engineering,Bangalore,99000.0,Active
6,Frank,Engineering,Chennai,115000.0,Active
10,Jack,HR,Chennai,58000.0,Active


####Ex 5→ MERGE - the upsert pattern (most important Delta operation)
MERGE is the single most used Delta Lake operation in real DE jobs. It inserts new rows AND updates existing ones in one atomic operation. This is how every incremental load pipeline is built.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

spark.sql("SELECT * FROM employees").display()

# Incoming batch — mix of updates and new records
incoming = spark.createDataFrame([
    (1,  "Alice",   "Engineering", "Delhi",    108000.0, "Active"),  # salary update
    (3,  "Charlie", "Engineering", "Hyderabad", 92000.0, "Active"),  # city + salary change
    (6,  "Frank",   "Engineering", "Chennai",  105000.0, "Active"),  # salary update
    (9,  "Ivy",     "Engineering", "Pune",      91000.0, "Active"),  # NEW employee
    (10, "Jack",    "HR",          "Chennai",   58000.0, "Active"),  # NEW employee
], schema)

# MERGE — upsert
delta_tbl = DeltaTable.forName(spark, "employees")

delta_tbl.alias("target") \
         .merge(incoming.alias("source"), "target.emp_id = source.emp_id") \
         .whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()      

print("After merge: ")
spark.sql("SELECT * FROM employees").display()

emp_id,name,dept,city,salary,status
2,Bob,Marketing,Mumbai,72000.0,Active
4,Diana,HR,Delhi,65000.0,Active
5,Eve,Marketing,Mumbai,78000.0,Active
7,Grace,HR,Bangalore,61000.0,Active
8,Henry,Marketing,Delhi,74000.0,Active
1,Alice,Engineering,Delhi,108000.0,Active
3,Charlie,Engineering,Hyderabad,92000.0,Active
6,Frank,Engineering,Chennai,105000.0,Active
9,Ivy,Engineering,Pune,91000.0,Active
10,Jack,HR,Chennai,58000.0,Active


After merge: 


emp_id,name,dept,city,salary,status
2,Bob,Marketing,Mumbai,72000.0,Active
4,Diana,HR,Delhi,65000.0,Active
5,Eve,Marketing,Mumbai,78000.0,Active
7,Grace,HR,Bangalore,61000.0,Active
8,Henry,Marketing,Delhi,74000.0,Active
3,Charlie,Engineering,Hyderabad,92000.0,Active
6,Frank,Engineering,Chennai,105000.0,Active
1,Alice,Engineering,Delhi,108000.0,Active
9,Ivy,Engineering,Pune,91000.0,Active
10,Jack,HR,Chennai,58000.0,Active


####Ex 6→ MERGE with conditional logic - soft delete and SCD Type 2
Advanced MERGE patterns used in real warehouses. Soft delete marks rows as inactive instead of physically removing them. These patterns appear in every mature DE pipeline.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, lit

# Reset table
df.write.format("delta").mode("overwrite").saveAsTable("employees")

delta_tbl = DeltaTable.forName(spark, "employees")

# ── Pattern 1: Selective update — only update salary if it increased
incoming_updates = spark.createDataFrame([
    (1, "Alice",   "Engineering", "Delhi",    90000.0, "Active"),  # LOWER salary — should NOT update
    (2, "Bob",     "Marketing",   "Mumbai",   85000.0, "Active"),  # HIGHER salary — should update
    (6, "Frank",   "Engineering", "Chennai",  120000.0,"Active"),  # HIGHER salary — should update
], schema)

delta_tbl.alias("t") \
         .merge(incoming_updates.alias("s"),"t.emp_id = s.emp_id") \
         .whenMatchedUpdate(
    condition = col("s.salary") > col("t.salary"),   # only update if new salary is higher
    set = {"salary": col("s.salary")}
).execute()

print("After conditional MERGE (only salary increases applied):")
spark.sql("SELECT emp_id, name, salary FROM employees ORDER BY emp_id").show()

# ── Pattern 2: Soft delete — mark inactive instead of deleting
# Reset
df.write.format("delta").mode("overwrite").saveAsTable("employees")
delta_tbl = DeltaTable.forName(spark, "employees")

resigned = spark.createDataFrame([(4,),(7,)], ["emp_id"])  # Diana and Grace resigned

delta_tbl.alias("t") \
         .merge(resigned.alias("s"),"t.emp_id = s.emp_id") \
         .whenMatchedUpdate(set = {"status": lit("Inactive")}
).execute()

print("After soft delete:")
spark.sql("SELECT emp_id, name, status FROM employees ORDER BY emp_id").show()
print("Active employees:", spark.sql("SELECT COUNT(*) FROM employees WHERE status='Active'").collect()[0][0])

After conditional MERGE (only salary increases applied):
+------+-------+--------+
|emp_id|   name|  salary|
+------+-------+--------+
|     1|  Alice| 95000.0|
|     2|    Bob| 85000.0|
|     3|Charlie| 88000.0|
|     4|  Diana| 65000.0|
|     5|    Eve| 78000.0|
|     6|  Frank|120000.0|
|     7|  Grace| 61000.0|
|     8|  Henry| 74000.0|
+------+-------+--------+

After soft delete:
+------+-------+--------+
|emp_id|   name|  status|
+------+-------+--------+
|     1|  Alice|  Active|
|     2|    Bob|  Active|
|     3|Charlie|  Active|
|     4|  Diana|Inactive|
|     5|    Eve|  Active|
|     6|  Frank|  Active|
|     7|  Grace|Inactive|
|     8|  Henry|  Active|
+------+-------+--------+

Active employees: 6
